In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')


train_df = pd.read_csv("/content/drive/MyDrive/widsdatathon2023/train_data.csv")
test_df = pd.read_csv("/content/drive/MyDrive/widsdatathon2023/test_data.csv")

In [3]:
def preprocess(df, is_train=True, fit_imputer=None):
    """
    Preprocess the raw data:
    - Parse the dates into year, month, and day
    - Remove the original date column and any index columns
    - Perform one-hot encoding on categorical variables
    - Handle missing values (fill with the median)
    Return the processed DataFrame (features + labels; if it is a training set, it includes labels)
    """
    df = df.copy()
    #  startdate
    def fix_date(x):
        try:
            if x.count('/') == 2 and len(x.split('/')[0]) == 4:
                return pd.to_datetime(x, format='%Y/%m/%d')
            else:
                return pd.to_datetime(x, format='%m/%d/%y')
        except:
            return pd.NaT
    df['startdate'] = df['startdate'].apply(fix_date)
    df['year'] = df['startdate'].dt.year
    df['month'] = df['startdate'].dt.month
    df['day'] = df['startdate'].dt.day
    df = df.drop(['startdate'], axis=1)
    # Remove any index columns that may exist
    if 'index' in df.columns:
        df = df.drop(['index'], axis=1)

    # Exclusive OR encoding
    df = pd.get_dummies(df, dtype=int)

    # Separator labels
    if is_train:
        target = 'contest-tmp2m-14d__tmp2m'
        y = df[target].values
        X = df.drop(columns=[target])
    else:
        y = None
        X = df

    # Handling missing values: Imputing with the median
    if fit_imputer is None:
        imputer = SimpleImputer(strategy='median')
        X_imputed = imputer.fit_transform(X)
    else:
        X_imputed = fit_imputer.transform(X)
    X = pd.DataFrame(X_imputed, columns=X.columns)

    if is_train:
        return X, y, imputer
    else:
        return X


In [4]:
X_train_full, y_train_full, imputer = preprocess(train_df, is_train=True)

feature_names = X_train_full.columns.tolist()

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)


In [5]:
X_test_raw = preprocess(test_df, is_train=False, fit_imputer=imputer)

for col in feature_names:
    if col not in X_test_raw.columns:
        X_test_raw[col] = 0
X_test_raw = X_test_raw[feature_names]

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test_raw)

In [7]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
pred_lr_val = lr.predict(X_val_scaled)
print("Linear Regression:")
print(f"  RMSE: {rmse(y_val, pred_lr_val):.4f}")
print(f"  R²: {r2_score(y_val, pred_lr_val):.4f}")

Linear Regression:
  RMSE: 1.2918
  R²: 0.9829


In [8]:
# Ridge（Cross-validation to select alpha）
ridge = RidgeCV(alphas=np.logspace(-3, 3, 50))
ridge.fit(X_train_scaled, y_train)
pred_ridge_val = ridge.predict(X_val_scaled)
print(f"\nRidge (alpha={ridge.alpha_:.4f}):")
print(f"  RMSE: {rmse(y_val, pred_ridge_val):.4f}")
print(f"  R²: {r2_score(y_val, pred_ridge_val):.4f}")



Ridge (alpha=0.0391):
  RMSE: 1.2918
  R²: 0.9829


In [9]:
# Lasso
lasso = LassoCV(alphas=np.logspace(-3, 3, 30), max_iter=10000, cv=3, n_jobs=-1)
lasso.fit(X_train_scaled, y_train)
pred_lasso_val = lasso.predict(X_val_scaled)
print(f"\nLasso (alpha={lasso.alpha_:.4f}):")
print(f"  RMSE: {rmse(y_val, pred_lasso_val):.4f}")
print(f"  R²: {r2_score(y_val, pred_lasso_val):.4f}")


Lasso (alpha=0.0010):
  RMSE: 1.3067
  R²: 0.9825


In [11]:
pred_test = lasso.predict(X_test_scaled)

# submission file
if 'index' in test_df.columns:
    submission = pd.DataFrame({
        'index': test_df['index'],
        'contest-tmp2m-14d__tmp2m': pred_test
    })
else:
    submission = pd.DataFrame({
        'contest-tmp2m-14d__tmp2m': pred_test
    })
    submission.index.name = 'index'
    submission.index = range(len(submission))

submission.to_csv('submission.csv', index=False)
print("Prediction complete. The submitted file has been saved as submission.csv")

Prediction complete. The submitted file has been saved as submission.csv
